# Traffic classification · binary

## Libraries and device

The seed identifies the run and the device determines where tensors are processed.

In [ ]:
import os
import json
import joblib  # Save the scaler
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from tqdm import tqdm  # Progress bar

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# ── Run configuration ──
# The seed identifies the random choices in this run. A comparison across methods also
# depends on using the same training and test examples.
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")

## Tuned parameters

The JSON files store architecture and training choices by model. Layer counts and widths must match the checkpoint before its weights can be loaded.

In [ ]:
MODEL_DIR = "trained_models"
os.makedirs(MODEL_DIR, exist_ok=True)

with open("bayesian_tuning/best_hyperparameters_summary_binary.json", "r") as f:
    best_hyperparameters = json.load(f)

## Reference dataset

Each CSV row describes a flow. Swapping columns can preserve tensor shape while changing what each input means.

In [ ]:
TASK = "binary"

df = pd.read_csv("Normal+Attacks.csv")

X = df.drop(columns=["label"], errors="ignore")
X = X.select_dtypes(include="number")
y = df["label"].apply(lambda x: 0 if x.strip().lower() == "normal" else 1)

# ── Reference holdout ──
# Stratification aims to retain label proportions. Two rows from one capture can still land
# on opposite sides of this row-level split.
X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)

scaler = StandardScaler()

# ── Scale fitted on training data ──
# Each column has its own mean and scale. The subsequent transform reuses these values even
# when the test distribution differs.
X_train_scaled = scaler.fit_transform(X_train_raw)
X_test_scaled = scaler.transform(X_test_raw)

import joblib
os.makedirs("trained_models", exist_ok=True)
joblib.dump(scaler, "trained_models/scaler_binary.joblib")


# Features become float32. Binary loss receives one numeric target per example, while
# multiclass loss expects an integer class index.
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_raw.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test_raw.values, dtype=torch.float32)

test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
test_loader = DataLoader(test_dataset, batch_size=1024, shuffle=False)

N_FEATURES = X_train_scaled.shape[1]
print(f"Data pipeline complete! Scaler saved in 'trained_models'.")
print(f"Training examples ready: {X_train_tensor.shape[0]}")
print(f"Test examples held out: {X_test_tensor.shape[0]}")

## Architectures

B is batch size and F is feature count. The MLP receives [B, F]. Recurrent models traverse features as [B, F, 1]. The CNN receives [B, 1, F], with one channel. Here F indexes features of a flow.

In [ ]:

class MLP(nn.Module):
    def __init__(self, input_dim, config):
        super().__init__()
        layers = []
        previous = input_dim
        n_layers = config["hiperparametros"]["n_camadas"]
        activation = config["hiperparametros"]["ativacao"]
        
        for i in range(n_layers):
            units = config["hiperparametros"][f"unidades_{i}"]
            dropout = config["hiperparametros"][f"dropout_{i}"]
            layers.append(nn.Linear(previous, units))
            layers.append(nn.ReLU() if activation == "relu" else nn.Tanh())
            if dropout > 0: layers.append(nn.Dropout(dropout))
            previous = units
            
        layers.append(nn.Linear(previous, 1))
        self.rede = nn.Sequential(*layers)

    def forward(self, x): return self.rede(x)

# ── Recurrent layers ──
# The traversed axis contains the features of one flow. Column order therefore defines the
# sequence seen by the network.
class Recorrente(nn.Module):
    def __init__(self, cell_type, input_dim, config):
        super().__init__()
        self.cell_type = cell_type
        self.camadas = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        n_layers = config["hiperparametros"]["n_camadas"]
        previous = input_dim
        
        for i in range(n_layers):
            units = config["hiperparametros"][f"unidades_{i}"]
            dropout = config["hiperparametros"][f"dropout_{i}"]
            if cell_type == "RNN": layer = nn.RNN(previous, units, batch_first=True)
            elif cell_type == "GRU": layer = nn.GRU(previous, units, batch_first=True)
            else: layer = nn.LSTM(previous, units, batch_first=True)
            self.camadas.append(layer)
            self.dropouts.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())
            previous = units
            
        self.densa = nn.Linear(previous, 64)
        self.relu = nn.ReLU()
        self.saida = nn.Linear(64, 1)

    def forward(self, x):
        for i, (layer, drop) in enumerate(zip(self.camadas, self.dropouts)):
            out, _ = layer(x)
            out = drop(out)
            if i < len(self.camadas) - 1: x = out
            else: x = out[:, -1, :]
        return self.saida(self.relu(self.densa(x)))

# ── Convolution over features ──
# The input has one channel. A filter combines neighboring features in CSV order, so column
# position matters.
class CNN(nn.Module):
    def __init__(self, input_channels, config):
        super().__init__()
        layers = []
        previous = input_channels
        
        n_blocos = config["hiperparametros"]["n_blocos_conv"]
        kernel_size = config["hiperparametros"]["kernel_size"]
        
        for i in range(n_blocos):
            filters = config["hiperparametros"][f"filtros_{i}"]
            layers.append(nn.Conv1d(previous, filters, kernel_size=kernel_size, padding=kernel_size//2))
            layers.append(nn.ReLU())
            layers.append(nn.MaxPool1d(kernel_size=2, ceil_mode=True))
            previous = filters
            
        self.conv = nn.Sequential(*layers)
        self.pool_global = nn.AdaptiveAvgPool1d(1)
        self.densa = nn.Linear(previous, 128)
        self.relu = nn.ReLU()
        
        self.dropout = nn.Dropout(config["hiperparametros"].get("dropout_final", 0.2))
        
        self.saida = nn.Linear(128, 1)

    def forward(self, x):
        x = self.conv(x)
        x = self.pool_global(x).squeeze(-1)
        return self.saida(self.dropout(self.relu(self.densa(x))))

print("Neural network classes loaded successfully!")

## Reference training

One batch produces a loss and a weight update. An epoch ends after the loader traverses the training set. The checkpoint allows the network to be restored for evaluation or robustness experiments.

In [ ]:
BATCH_SIZE = 256
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

EPOCHS = 50

def train_final_model(name, model, lr):
    print("\n" + "=" * 60)
    print(f"TRAINING: {name}")
    print("=" * 60)
    
    model = model.to(DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    
    for epoch in range(EPOCHS):
        model.train()
        training_progress = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
        
        for xb, y_batch in training_progress:
            xb, y_batch = xb.to(DEVICE), y_batch.to(DEVICE)
            if name in ["RNN", "GRU", "LSTM"]: xb = xb.unsqueeze(-1)
            elif name == "CNN": xb = xb.unsqueeze(-1).permute(0, 2, 1)
            
            optimizer.zero_grad()
            loss = criterion(model(xb).view(-1), y_batch)
            loss.backward()
            optimizer.step()
            training_progress.set_postfix(loss=f"{loss.item():.4f}")
            
    save_path = f"{MODEL_DIR}/{name}_binary.pt"
    torch.save({
        "state_dict": model.state_dict(),
        "n_classes": 2,
        "n_features": N_FEATURES
    }, save_path)
    print(f"Neural network {name} saved successfully to: {save_path}")
    return model

model_instances = {
    "MLP":  MLP(N_FEATURES, best_hyperparameters["MLP"]),
    "RNN":  Recorrente("RNN", 1, best_hyperparameters["RNN"]),
    "CNN":  CNN(1, best_hyperparameters["CNN"]),
    "GRU":  Recorrente("GRU", 1, best_hyperparameters["GRU"]),
    "LSTM": Recorrente("LSTM", 1, best_hyperparameters["LSTM"])
}

trained_models = {}
for name in ["MLP", "RNN", "CNN", "GRU", "LSTM"]:
    model_learning_rate = best_hyperparameters[name]["hiperparametros"]["learning_rate"]
    trained_models[name] = train_final_model(name, model_instances[name], model_learning_rate)

## Test evaluation

In the confusion matrix, rows are true labels and columns are predictions. A row sum gives class support.

In [ ]:

FIGURES_DIR = "../graphics"
os.makedirs(FIGURES_DIR, exist_ok=True)

for name, model in trained_models.items():
    model.eval()
    predicoes_completas = []
    
    with torch.no_grad():
        for xb_t, _ in test_loader:
            xb_t = xb_t.to(DEVICE)
            if name in ["RNN", "GRU", "LSTM"]: 
                xb_t = xb_t.unsqueeze(-1)
            elif name == "CNN": 
                xb_t = xb_t.unsqueeze(-1).permute(0, 2, 1)
            
            logits = model(xb_t).view(-1)
            probs = torch.sigmoid(logits)
            preds_binarias = (probs > 0.5).int().cpu().numpy()
            predicoes_completas.append(preds_binarias)
            
    y_pred = np.concatenate(predicoes_completas)
    y_true = y_test_raw.values # Resgata o gabarito original do scikit-learn
    
    print(f"\nPERFORMANCE REPORT: {name}")
    print(classification_report(y_true, y_pred, target_names=["Normal", "Attack"]))
    
    cm = confusion_matrix(y_true, y_pred)
    
    # ── Row percentages ──
    # Each row sums to approximately 100%. Its diagonal entry gives recall for that class,
    # even when its support is small.
    cm_norm = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis] * 100
    
    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm_norm, 
        annot=True,          # Add values as text inside each cell
        fmt=".2f",           # Limit display to two decimal places
        cmap="Blues",        # Blue color palette for the binary task
        xticklabels=["Normal", "Attack"], 
        yticklabels=["Normal", "Attack"]
    )
    
    plt.title(f"Confusion Matrix - {name} Binary")
    plt.xlabel("Model prediction (%)")
    plt.ylabel("True class (%)")
    plt.tight_layout()
    
    figure_path = os.path.join(FIGURES_DIR, f"binary_classification_original_confusion_matrix_{name}.png")
    plt.savefig(figure_path, dpi=150)
    plt.close() # Close the figure to save VS Code memory
    print(f"Confusion-matrix figure exported to: {figure_path}")

## Reading the results

## Cleaned-data experiment

Groups are keyed by feature values. Two records with identical vectors stay on the same side of the split even when their labels disagree. This controls one source of leakage. Evaluating generalization to a new capture requires capture identifiers.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

CLEANED_SEED = 42
CLEANED_BATCH_SIZE = 256
CLEANED_EPOCHS = 50
CLEANED_THRESHOLD = 0.5
CLEANED_DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
np.random.seed(CLEANED_SEED)
torch.manual_seed(CLEANED_SEED)

DATA_VERSION = "cleaned_data"
CLEANED_RESULTS_DIR = "treated_binary_classification"
CLEANED_MODEL_DIR = os.path.join(CLEANED_RESULTS_DIR, "models")
CLEANED_GRAPHICS_DIR = os.path.join("..", "graphics")
os.makedirs(CLEANED_MODEL_DIR, exist_ok=True)
os.makedirs(CLEANED_GRAPHICS_DIR, exist_ok=True)

with open("bayesian_tuning/best_hyperparameters_summary_binary.json", "r") as file:
    best_hyperparameters_cleaned = json.load(file)

df_original_exp = pd.read_csv("Normal+Attacks.csv")
linhas_originais_exp = len(df_original_exp)
contagem_original_exp = df_original_exp["label"].value_counts().to_dict()

df_cleaned = df_original_exp.drop(columns=["fwd_seg_size_min"], errors="ignore").copy()
X_all_experiments = df_cleaned.drop(columns=["label"]).select_dtypes(include="number")
feature_names_exp = X_all_experiments.columns.tolist()

y_all_experiments = df_cleaned["label"].astype(str).str.strip().str.lower().ne("normal").astype("int64")
cleaned_data = X_all_experiments.copy()
cleaned_data["target"] = y_all_experiments.to_numpy()

# ── Duplicate rows ──
# The target is already in the table. Deduplication compares complete records. Grouping later
# handles identical feature vectors retained because their labels differ.
cleaned_data = cleaned_data.drop_duplicates().reset_index(drop=True)
rows_after_cleaning = len(cleaned_data)
X_cleaned = cleaned_data[feature_names_exp]
y_cleaned = cleaned_data["target"].to_numpy(dtype="float32")

feature_groups = pd.util.hash_pandas_object(X_cleaned, index=False).to_numpy()
splitter_exp = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=CLEANED_SEED)
cleaned_train_indices, cleaned_test_indices = next(
    splitter_exp.split(X_cleaned, y_cleaned, groups=feature_groups)
)
X_train_raw_exp = X_cleaned.iloc[cleaned_train_indices]
X_test_raw_exp = X_cleaned.iloc[cleaned_test_indices]
y_train_exp = y_cleaned[cleaned_train_indices]
y_test_exp = y_cleaned[cleaned_test_indices]

scaler_cleaned = StandardScaler()
X_train_exp = scaler_cleaned.fit_transform(X_train_raw_exp)
X_test_exp = scaler_cleaned.transform(X_test_raw_exp)
N_FEATURES_CLEANED = X_train_exp.shape[1]
X_train_tensor_exp = torch.tensor(X_train_exp, dtype=torch.float32)
X_test_tensor_exp = torch.tensor(X_test_exp, dtype=torch.float32)
y_train_tensor_exp = torch.tensor(y_train_exp, dtype=torch.float32)
y_test_tensor_exp = torch.tensor(y_test_exp, dtype=torch.float32)
train_loader_exp = DataLoader(
    TensorDataset(X_train_tensor_exp, y_train_tensor_exp),
    batch_size=CLEANED_BATCH_SIZE,
    shuffle=True,
    generator=torch.Generator().manual_seed(CLEANED_SEED),
)
test_loader_exp = DataLoader(
    TensorDataset(X_test_tensor_exp, y_test_tensor_exp), batch_size=1024, shuffle=False
)

class CleanedBinaryMLP(nn.Module):
    def __init__(self, input_dim, config):
        super().__init__()
        hp = config["hiperparametros"]
        layers = []
        previous = input_dim
        for index in range(hp["n_camadas"]):
            units = hp[f"unidades_{index}"]
            dropout = hp[f"dropout_{index}"]
            layers.extend([nn.Linear(previous, units), nn.ReLU() if hp["ativacao"] == "relu" else nn.Tanh()])
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            previous = units
        layers.append(nn.Linear(previous, 1))
        self.rede = nn.Sequential(*layers)

    def forward(self, x):
        return self.rede(x)

# ── Recurrent layers ──
# The traversed axis contains the features of one flow. Column order therefore defines the
# sequence seen by the network.
class CleanedBinaryRecurrentModel(nn.Module):
    def __init__(self, architecture, config):
        super().__init__()
        hp = config["hiperparametros"]
        self.camadas = nn.ModuleList()
        self.dropouts = nn.ModuleList()
        previous = 1
        for index in range(hp["n_camadas"]):
            units = hp[f"unidades_{index}"]
            dropout = hp[f"dropout_{index}"]
            self.camadas.append({"RNN": nn.RNN, "GRU": nn.GRU, "LSTM": nn.LSTM}[architecture](previous, units, batch_first=True))
            self.dropouts.append(nn.Dropout(dropout) if dropout > 0 else nn.Identity())
            previous = units
        self.densa = nn.Linear(previous, 64)
        self.saida = nn.Linear(64, 1)

    def forward(self, x):
        for index, (layer, dropout) in enumerate(zip(self.camadas, self.dropouts)):
            output, _ = layer(x)
            output = dropout(output)
            x = output if index < len(self.camadas) - 1 else output[:, -1, :]
        return self.saida(torch.relu(self.densa(x)))

# ── Convolution over features ──
# The input has one channel. A filter combines neighboring features in CSV order, so column
# position matters.
class CleanedBinaryCNN(nn.Module):
    def __init__(self, config):
        super().__init__()
        hp = config["hiperparametros"]
        layers = []
        previous = 1
        kernel_size = hp["kernel_size"]
        for index in range(hp["n_blocos_conv"]):
            filters = hp[f"filtros_{index}"]
            layers.extend([nn.Conv1d(previous, filters, kernel_size, padding=kernel_size // 2), nn.ReLU(), nn.MaxPool1d(2, ceil_mode=True)])
            previous = filters
        self.conv = nn.Sequential(*layers)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.densa = nn.Linear(previous, 128)
        self.dropout = nn.Dropout(hp.get("dropout_final", 0.2))
        self.saida = nn.Linear(128, 1)

    def forward(self, x):
        x = self.pool(self.conv(x)).squeeze(-1)
        return self.saida(self.dropout(torch.relu(self.densa(x))))


def prepare_binary_input(name, xb):
    if name in ("RNN", "GRU", "LSTM"):
        return xb.unsqueeze(-1)
    if name == "CNN":
        return xb.unsqueeze(-1).permute(0, 2, 1)
    return xb


construtores_exp = {
    "MLP": lambda: CleanedBinaryMLP(N_FEATURES_CLEANED, best_hyperparameters_cleaned["MLP"]),
    "RNN": lambda: CleanedBinaryRecurrentModel("RNN", best_hyperparameters_cleaned["RNN"]),
    "CNN": lambda: CleanedBinaryCNN(best_hyperparameters_cleaned["CNN"]),
    "GRU": lambda: CleanedBinaryRecurrentModel("GRU", best_hyperparameters_cleaned["GRU"]),
    "LSTM": lambda: CleanedBinaryRecurrentModel("LSTM", best_hyperparameters_cleaned["LSTM"]),
}

joblib.dump(scaler_cleaned, os.path.join(CLEANED_MODEL_DIR, "scaler_binary.joblib"))
training_metrics = {}

for name, build_model in construtores_exp.items():
    print(f"\nTraining {name} with cleaned data (binary) ({CLEANED_DEVICE})")
    model = build_model().to(CLEANED_DEVICE)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=best_hyperparameters_cleaned[name]["hiperparametros"]["learning_rate"])

    for epoch in range(CLEANED_EPOCHS):
        model.train()
        for xb, y_batch in tqdm(train_loader_exp, desc=f"{name} {epoch + 1}/{CLEANED_EPOCHS}", leave=False):
            xb, y_batch = xb.to(CLEANED_DEVICE), y_batch.to(CLEANED_DEVICE)
            xb = prepare_binary_input(name, xb)
            optimizer.zero_grad()
            loss = criterion(model(xb).view(-1), y_batch)
            loss.backward()
            optimizer.step()

    model.eval()
    experiment_probabilities = []
    with torch.no_grad():
        for xb, _ in test_loader_exp:
            xb = prepare_binary_input(name, xb.to(CLEANED_DEVICE))
            experiment_probabilities.extend(torch.sigmoid(model(xb).view(-1)).cpu().numpy().tolist())

    y_prob_exp = np.asarray(experiment_probabilities)
    y_pred_exp = (y_prob_exp >= CLEANED_THRESHOLD).astype("int64")
    y_true_exp = y_test_exp.astype("int64")
    relatorio_exp = classification_report(y_true_exp, y_pred_exp, labels=[0, 1], target_names=["Normal", "Attack"], output_dict=True, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true_exp, y_pred_exp, labels=[0, 1]).ravel()
    training_metrics[name] = {
        "threshold": CLEANED_THRESHOLD,
        "accuracy": float(accuracy_score(y_true_exp, y_pred_exp)),
        "false_positive_rate": float(fp / (fp + tn)) if fp + tn else 0.0,
        "specificity": float(tn / (tn + fp)) if tn + fp else 0.0,
        "attack_recall": float(tp / (tp + fn)) if tp + fn else 0.0,
        "attack_precision": float(tp / (tp + fp)) if tp + fp else 0.0,
        "classification_report": relatorio_exp,
    }
    print(classification_report(y_true_exp, y_pred_exp, labels=[0, 1], target_names=["Normal", "Attack"], zero_division=0))
    print(f"False-positive rate among Normal examples: {training_metrics[name]['false_positive_rate']:.2%}")

    torch.save({
        "state_dict": model.state_dict(),
        "n_classes": 2,
        "n_features": N_FEATURES_CLEANED,
        "feature_names": feature_names_exp,
        "threshold": CLEANED_THRESHOLD,
    }, os.path.join(CLEANED_MODEL_DIR, f"{name}_binary.pt"))

    matriz_exp = confusion_matrix(y_true_exp, y_pred_exp, labels=[0, 1])
    normalized_confusion_matrix = matriz_exp.astype(float) / np.maximum(matriz_exp.sum(axis=1, keepdims=True), 1) * 100
    plt.figure(figsize=(6, 5))
    sns.heatmap(normalized_confusion_matrix, annot=True, fmt=".2f", cmap="Blues", xticklabels=["Normal", "Attack"], yticklabels=["Normal", "Attack"])
    plt.title(f"Confusion Matrix - {name} Binary (deduplicado)")
    plt.xlabel("Model prediction (%)")
    plt.ylabel("True class (%)")
    plt.tight_layout()
    plt.savefig(os.path.join(CLEANED_GRAPHICS_DIR, f"binary_classification_cleaned_confusion_matrix_{name}.png"), dpi=150)
    plt.close()

with open(os.path.join(CLEANED_RESULTS_DIR, "training_metrics.json"), "w", encoding="utf-8") as file:
    json.dump(training_metrics, file, indent=2, ensure_ascii=False)

training_summary = {
    "version": DATA_VERSION,
    "seed": CLEANED_SEED,
    "epochs": CLEANED_EPOCHS,
    "threshold": CLEANED_THRESHOLD,
    "original_rows": linhas_originais_exp,
    "duplicates_removed": linhas_originais_exp - rows_after_cleaning,
    "rows_after_cleaning": rows_after_cleaning,
    "feature_vectors_with_conflicting_labels": int((cleaned_data.groupby(feature_names_exp, dropna=False)["target"].nunique() > 1).sum()),
    "removed_column": "fwd_seg_size_min",
    "binary_class_counts_after_cleaning": {"Normal": int((y_cleaned == 0).sum()), "Attack": int((y_cleaned == 1).sum())},
    "features": feature_names_exp,
    "train_rows": int(len(y_train_exp)),
    "test_rows": int(len(y_test_exp)),
    "metrics": {name: {key: values[key] for key in ("accuracy", "false_positive_rate", "specificity", "attack_recall", "attack_precision")} for name, values in training_metrics.items()},
}
with open(os.path.join(CLEANED_RESULTS_DIR, "training_summary.json"), "w", encoding="utf-8") as file:
    json.dump(training_summary, file, indent=2, ensure_ascii=False)

print(f"\nBinary classification complete. Current results in: {CLEANED_RESULTS_DIR}")